# Seguimiento de avance - "economía popular"

Este notebook es **liviano y separado** del que corre el scraper: no instala Playwright ni abre un navegador. Solo *lee* los checkpoints que el otro notebook va guardando en tu Drive, arma un Excel con el avance actual, y corre el EDA.

Como solo lee (nunca escribe en los checkpoints), lo podés tener abierto en **otro runtime de Colab** en paralelo, sin miedo a pisarle nada al scraper que sigue corriendo en la otra pestaña/notebook.

Ejecutá las celdas en orden (Shift+Enter).

## Paso 1 - Instalar dependencias (livianas, sin Playwright)

In [ ]:
!pip install -q pandas openpyxl matplotlib


## Paso 2 - Montar Google Drive

Usa la misma carpeta `economia_popular` que ya está usando el notebook del scraper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA_PROYECTO = "/content/drive/MyDrive/economia_popular"
print("Leyendo checkpoints desde:", CARPETA_PROYECTO)


## Paso 3 - Armar el Excel de avance

Lee los checkpoints tal como estén en este momento y guarda `corpus_economia_popular_avance.xlsx` en la carpeta de Drive. Podés volver a correr esta celda cuantas veces quieras - cada vez toma una "foto" fresca de cómo va el scraping.

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

CHECKPOINT_DIR = Path(f"{CARPETA_PROYECTO}/checkpoints")

FECHA_DESDE = datetime(2019, 9, 1)
FECHA_HASTA = datetime(2023, 9, 30)

PATRON_MENCION = re.compile(r"econom[ií]as?\s+popular(?:es)?", re.IGNORECASE)


def cargar_checkpoint(nombre, default=None):
    ruta = CHECKPOINT_DIR / nombre
    if ruta.exists():
        return json.loads(ruta.read_text(encoding="utf-8"))
    return default if default is not None else {}


def _parsear_fecha(fecha_str):
    if not fecha_str:
        return None
    if isinstance(fecha_str, datetime):
        return fecha_str
    formatos = ("%Y-%m-%dT%H:%M:%S.%fZ", "%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%d")
    for fmt in formatos:
        try:
            return datetime.strptime(fecha_str, fmt)
        except ValueError:
            continue
    try:
        return datetime.fromisoformat(fecha_str.replace("Z", "+00:00")).replace(tzinfo=None)
    except Exception:
        return None


def filtrar_por_fecha(articulos, desde, hasta):
    filtrados = []
    descartados_sin_fecha = 0
    for a in articulos:
        fecha_raw = a.get("fecha")
        fecha = fecha_raw if isinstance(fecha_raw, datetime) else _parsear_fecha(fecha_raw)
        if fecha is None:
            descartados_sin_fecha += 1
            continue
        a["fecha"] = fecha
        if desde <= fecha <= hasta:
            filtrados.append(a)
    if descartados_sin_fecha:
        print(f"(aviso) {descartados_sin_fecha} artículos sin fecha parseable quedaron afuera del filtro.")
    return filtrados


def menciona_economia_popular(articulo):
    campos = [articulo.get("texto_completo"), articulo.get("titulo"), articulo.get("volanta_bajada")]
    for campo in campos:
        if campo and PATRON_MENCION.search(campo):
            return True
    return False


def exportar_avance_parcial(nombre_archivo="corpus_economia_popular_avance.xlsx"):
    """Arma un Excel con todo lo que hay guardado en los checkpoints HASTA
    AHORA (los va escribiendo el otro notebook, el que scrapea), sin
    esperar a que termine el proceso completo. Esta función solo LEE esos
    checkpoints, nunca escribe en ellos, así que no interfiere con el
    scraper que puede estar corriendo en paralelo en otro runtime."""
    estado_p12 = cargar_checkpoint("pagina12_estado.json", {"offset": 0, "articulos": []})
    articulos_p12 = estado_p12["articulos"]
    textos_p12 = cargar_checkpoint("pagina12_textos.json", {})
    for art in articulos_p12:
        art["texto_completo"] = textos_p12.get(art["link"])
    articulos_p12 = filtrar_por_fecha(articulos_p12, FECHA_DESDE, FECHA_HASTA)

    # La Nación: solo las notas recuperadas con la sesión de la suscripción.
    estado_log = cargar_checkpoint("lanacion_logueado.json", {"articulos": [], "fallidos": {}})
    articulos_ln = [a for a in estado_log["articulos"] if menciona_economia_popular(a)]
    articulos_ln = filtrar_por_fecha(articulos_ln, FECHA_DESDE, FECHA_HASTA)

    todos = articulos_p12 + articulos_ln
    print(f"Página/12: {len(articulos_p12)} | La Nación: {len(articulos_ln)} | Total: {len(todos)}")

    df = pd.DataFrame(todos)
    columnas = ["medio", "titulo", "fecha", "autor", "seccion", "volanta_bajada", "texto_completo", "link"]
    for col in columnas:
        if col not in df.columns:
            df[col] = None
    df = df[columnas].sort_values(["medio", "fecha"])

    ruta = Path(f"{CARPETA_PROYECTO}/{nombre_archivo}")
    df.to_excel(ruta, index=False)
    print(f"Guardado en: {ruta}")
    return df


exportar_avance_parcial()


## Paso 4 - EDA sobre el avance actual

In [ ]:
"""
EDA rápido del corpus "economía popular" - para la reunión de avances.

Pensado para correr en el mismo notebook de Colab donde ya tenés todo
armado (reusa CARPETA_PROYECTO), o de forma standalone apuntando
RUTA_CORPUS a donde tengas el Excel.

Qué muestra:
1. Resumen general (cuántas notas por medio, rango de fechas, cobertura).
2. Evolución temporal de menciones por trimestre y medio - lo más
   relevante para la hipótesis del cambio de sentido durante el
   Frente de Todos.
3. Distribución por sección.
4. Autores con más notas.
5. Cantidad de veces que aparece la frase por nota, y longitud de los
   textos.
6. Notas por año y medio (para comparar cobertura entre medios).

Ojo: si corrés esto mientras el scraper de La Nación todavía no terminó,
los números de La Nación son parciales - aclaralo en la reunión.
"""

import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------

# Si estás en el notebook de Colab, dejalo así (usa la carpeta ya montada).
# Si corrés esto suelto, cambiá esta línea por la ruta a tu Excel.
try:
    RUTA_CORPUS = f"{CARPETA_PROYECTO}/corpus_economia_popular_avance.xlsx"
except NameError:
    RUTA_CORPUS = "corpus_economia_popular_avance.xlsx"

PATRON_MENCION = re.compile(r"econom[ií]as?\s+popular(?:es)?", re.IGNORECASE)


# ---------------------------------------------------------------------------
# 1. CARGA Y RESUMEN GENERAL
# ---------------------------------------------------------------------------

df = pd.read_excel(RUTA_CORPUS)
df["fecha"] = pd.to_datetime(df["fecha"])

print("=" * 60)
print("RESUMEN GENERAL")
print("=" * 60)
print(f"Total de notas: {len(df)}")
print(df["medio"].value_counts().to_string())
print(f"\nRango de fechas: {df['fecha'].min().date()} a {df['fecha'].max().date()}")
print(f"Notas sin texto completo: {df['texto_completo'].isna().sum()} "
      f"({df['texto_completo'].isna().mean():.1%})")


# ---------------------------------------------------------------------------
# 2. EVOLUCIÓN TEMPORAL (lo más relevante para la hipótesis)
# ---------------------------------------------------------------------------

df["trimestre"] = df["fecha"].dt.to_period("Q").astype(str)
evolucion = df.groupby(["trimestre", "medio"]).size().unstack(fill_value=0)

evolucion.plot(kind="line", marker="o", figsize=(13, 5),
                title='Menciones de "economía popular" por trimestre y medio')
plt.ylabel("Cantidad de notas")
plt.xlabel("Trimestre")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 3. NOTAS POR AÑO Y MEDIO
# ---------------------------------------------------------------------------

df["anio"] = df["fecha"].dt.year
por_anio = df.groupby(["anio", "medio"]).size().unstack(fill_value=0)

por_anio.plot(kind="bar", figsize=(10, 5), title="Notas por año y medio")
plt.ylabel("Cantidad de notas")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 4. DISTRIBUCIÓN POR SECCIÓN
# ---------------------------------------------------------------------------

top_secciones = df["seccion"].value_counts().head(15)

top_secciones.plot(kind="barh", figsize=(8, 6), title="Notas por sección (top 15)")
plt.gca().invert_yaxis()
plt.xlabel("Cantidad de notas")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 5. AUTORES CON MÁS NOTAS
# ---------------------------------------------------------------------------

autores = df["autor"].dropna()
autores = autores[autores.str.strip() != ""]
top_autores = autores.value_counts().head(15)

top_autores.plot(kind="barh", figsize=(8, 6), title="Autores con más notas (top 15)")
plt.gca().invert_yaxis()
plt.xlabel("Cantidad de notas")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6. MENCIONES POR NOTA Y LONGITUD DE TEXTO
# ---------------------------------------------------------------------------

def contar_menciones(texto):
    if not isinstance(texto, str):
        return 0
    return len(PATRON_MENCION.findall(texto))


def contar_palabras(texto):
    if not isinstance(texto, str):
        return 0
    return len(texto.split())


df["n_menciones"] = df["texto_completo"].apply(contar_menciones)
df["n_palabras"] = df["texto_completo"].apply(contar_palabras)

print("\n" + "=" * 60)
print("MENCIONES Y LONGITUD DE TEXTO (por medio)")
print("=" * 60)
print(df.groupby("medio")[["n_menciones", "n_palabras"]].describe().to_string())

df["n_menciones"].plot(kind="hist", bins=20, figsize=(8, 5),
                        title='Distribución de menciones de "economía popular" por nota')
plt.xlabel("Cantidad de menciones en la nota")
plt.tight_layout()
plt.show()

print("\nListo - estos gráficos/números están pensados para pegar capturas "
      "en la presentación de mañana.")


## Paso 5 - Quién nombra el término y con qué connotación (primer pase automático)

Para cada mención de "economía popular" en el texto: recorta el fragmento de contexto, busca un verbo de atribución cerca (`afirmó`, `sostuvo`, `señaló`...) y el nombre propio más cercano como **actor candidato**.

Esto es una heurística genérica (nombre propio más cercano), útil para priorizar lectura, pero no reemplaza el codebook real del proyecto (Tarea 1/2, Paso 6). Guarda `fragmentos_economia_popular.xlsx` en Drive con una fila por mención.

In [ ]:
"""
Extracción automática de fragmentos alrededor de cada mención de
"economía popular" - primer paso para la codificación de quién nombra el
término y con qué connotación.

Por qué existe este script: según lo que definimos con el proyecto, la
codificación final (tipo de actor, sentido/marco) necesita un codebook
que todavía hay que armar con Gabriela. Pero juntar cada mención
con su contexto - y una primera pista automática de quién podría estar
hablando ahí - no depende de ese codebook y sirve para tener el material
ya recortado y listo cuando se haga la codificación manual.

Qué hace, para cada mención de "economía popular" en el texto completo:
1. Recorta un fragmento de contexto alrededor (antes y después).
2. Busca un verbo de atribución de discurso cerca ("afirmó", "sostuvo",
   "señaló", etc.) - si aparece uno, es señal de que alguien está siendo
   citado o parafraseado ahí, no que es la voz del propio medio.
3. Busca el nombre propio más cercano (heurística simple de mayúsculas,
   NO es NER real) como actor candidato - hay que confirmarlo a mano.

El punto 3 no reemplaza la codificación cualitativa con codebook - es
una pista para acelerarla, no la respuesta.

Uso:
    python3 extraer_fragmentos.py
Lee el corpus (RUTA_CORPUS más abajo) y guarda
fragmentos_economia_popular.xlsx con una fila por mención.
"""

import re
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------

try:
    RUTA_CORPUS = f"{CARPETA_PROYECTO}/corpus_economia_popular_avance.xlsx"
except NameError:
    RUTA_CORPUS = "corpus_economia_popular_avance.xlsx"

try:
    RUTA_SALIDA = f"{CARPETA_PROYECTO}/fragmentos_economia_popular.xlsx"
except NameError:
    RUTA_SALIDA = "fragmentos_economia_popular.xlsx"

PATRON_MENCION = re.compile(r"econom[ií]as?\s+popular(?:es)?", re.IGNORECASE)

# Caracteres de contexto antes/después de cada mención (~2-3 oraciones)
VENTANA_CHARS = 300

# Verbos de atribución de discurso: si aparecen cerca de la mención, es
# señal de que alguien está siendo citado o parafraseado ahí.
VERBOS_ATRIBUCION = [
    "dijo", "afirmó", "sostuvo", "señaló", "indicó", "aseguró", "explicó",
    "consideró", "planteó", "advirtió", "remarcó", "subrayó", "destacó",
    "manifestó", "opinó", "sugirió", "definió", "describió", "criticó",
    "denunció", "reclamó", "pidió", "propuso", "anunció", "declaró",
    "expresó", "resaltó", "insistió", "reconoció", "admitió",
]
PATRON_ATRIBUCION = re.compile(r"\b(" + "|".join(VERBOS_ATRIBUCION) + r")\b", re.IGNORECASE)

# Nombre propio candidato: secuencia de 2+ palabras con mayúscula inicial.
# Heurística simple (no es NER), pensada para acotar quién puede estar
# hablando - hay que confirmarlo a mano en la codificación.
PATRON_NOMBRE_PROPIO = re.compile(
    r"\b([A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+){1,3})\b"
)


# ---------------------------------------------------------------------------
# EXTRACCIÓN
# ---------------------------------------------------------------------------

def _ventana(texto, inicio, fin):
    desde = max(0, inicio - VENTANA_CHARS)
    hasta = min(len(texto), fin + VENTANA_CHARS)
    return texto[desde:hasta].replace("\n", " ").strip(), desde


def _actor_candidato(fragmento, pos_mencion_en_fragmento):
    """Heurística: si hay un verbo de atribución cerca, preferimos el
    nombre propio que aparece ANTES de ese verbo (patrón típico:
    "Fulano afirmó que la economía popular..."). Si no hay verbo,
    devolvemos el nombre propio más cercano a la mención como candidato
    débil (menos confiable, hay que revisar a mano)."""
    m_verbo = PATRON_ATRIBUCION.search(fragmento)
    nombres = list(PATRON_NOMBRE_PROPIO.finditer(fragmento))
    if not nombres:
        return None, (m_verbo.group(1) if m_verbo else None)

    if m_verbo:
        antes_del_verbo = [n for n in nombres if n.start() < m_verbo.start()]
        if antes_del_verbo:
            return antes_del_verbo[-1].group(1), m_verbo.group(1)

    mas_cercano = min(nombres, key=lambda n: abs(n.start() - pos_mencion_en_fragmento))
    return mas_cercano.group(1), (m_verbo.group(1) if m_verbo else None)


def extraer_fragmentos(df):
    filas = []
    for _, art in df.iterrows():
        texto = art.get("texto_completo")
        if not isinstance(texto, str) or not texto.strip():
            continue
        for m in PATRON_MENCION.finditer(texto):
            fragmento, desde = _ventana(texto, m.start(), m.end())
            pos_relativa = m.start() - desde
            actor, verbo = _actor_candidato(fragmento, pos_relativa)
            filas.append({
                "medio": art.get("medio"),
                "titulo": art.get("titulo"),
                "fecha": art.get("fecha"),
                "seccion": art.get("seccion"),
                "link": art.get("link"),
                "mencion_textual": m.group(0),
                "fragmento": fragmento,
                "actor_candidato": actor,
                "verbo_atribucion": verbo,
            })
    return pd.DataFrame(filas)


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

print(f"Leyendo corpus desde: {RUTA_CORPUS}")
df = pd.read_excel(RUTA_CORPUS)

fragmentos = extraer_fragmentos(df)
print(f"\nTotal de menciones encontradas: {len(fragmentos)} (en {df['texto_completo'].notna().sum()} notas con texto)")

fragmentos.to_excel(RUTA_SALIDA, index=False)
print(f"Guardado en: {RUTA_SALIDA}")

print("\n" + "=" * 60)
print("PARA LA REUNIÓN - pantallazo rápido (todo esto es provisorio)")
print("=" * 60)

print("\nMenciones con verbo de atribución cerca (posible cita/parafraseo):")
con_verbo = fragmentos["verbo_atribucion"].notna().sum()
print(f"  {con_verbo} de {len(fragmentos)} ({con_verbo / len(fragmentos):.1%})" if len(fragmentos) else "  (sin datos)")

print("\nActores candidatos más frecuentes (heurística, revisar a mano):")
print(fragmentos["actor_candidato"].value_counts().head(15).to_string())

print(
    "\nOjo: 'actor_candidato' es una pista automática para priorizar la "
    "lectura, no la codificación final. Esa codificación (tipo de actor, "
    "sentido/marco) se hace con el codebook del proyecto (ver "
    "analisis_emisores_marcos.py) o leyendo la nota completa."
)


## Paso 6 - Tarea 1 y 2: emisores y marcos de referencia (codebook de Daniela)

**Tarea 1 - tipo de emisor.** Busca en cada fragmento qué emisores aparecen y los clasifica según la taxonomía del documento de metodología:

1. **Estatal**: Presidente · Área social/desarrollo · Área económica/laboral · Subnacional (gobernadores, intendentes) · Otros poderes (Legislativo, Judicial).
2. **Organizaciones y movimientos**: UTEP y sus organizaciones (Movimiento Evita, Barrios de Pie, MTE, CCC, etc.) · Frente Patria Grande · Izquierda/no alineadas (Polo Obrero, MST) · Sindicatos tradicionales (CGT, CTA).
3. **Dirigentes sociales**: Grabois, Pérsico, "Gringo" Castro, "Peluca" Gramajo, "Chino" Navarro.
4. **Sector privado**: UIA, CAME, Sociedad Rural, cámaras, empresarios.
5. **Otros / medios / academia**: expertos, analistas, y las **notas de opinión / editoriales** (se detectan por la sección o el link de la nota).

**Tarea 2 - marco de referencia.** Cruza cada emisor con los 16 marcos del documento (organizaciones de la EP, movimientos sociales, sector, políticas sociales/laborales, trabajadores informales, pobres, humildes, piqueteros, derechos, IFE, Potenciar Trabajo, trabajo/trabajadores, asistencia social, informalidad/precarización, conflicto/seguridad/orden público, integración económica/desarrollo productivo). Los marcos 12 a 16 se detectan por cualquiera de los términos que el documento lista en *Comprende*.

Es codificación por diccionario (frases/nombres, tolerante a tildes y mayúsculas; no NLP real). Los patrones genéricos (*sector*, *derechos*, *asistencia*, *marcha*, *trabajador*, *empresarios*, *el presidente*) pueden dar falsos positivos, y algunos marcos se superponen a propósito (p. ej. 05 y 14, o 08 y 15), tal como están en el documento. Para sumar o corregir un emisor o un marco, se edita el diccionario `EMISORES` / `MARCOS` de la celda de abajo.

Guarda `fragmentos_codificados.xlsx` (una fila por mención, con `emisores_detectados`, `tipo_emisor`, `subtipo_emisor`, `marcos_detectados`) y `tabla_emisores_marcos.xlsx` (hojas `tipo_x_marco`, `subtipo_x_marco`, `emisor_x_marco`) en Drive.

In [ ]:
"""
Tarea 1 y Tarea 2 del codebook: emisores y marcos de referencia de
"economía popular", según la taxonomía de Daniela (documento de
metodología: tipo de emisor + objeto directo / marco de referencia).

Tarea 1: para cada mención de "economía popular" (con su fragmento de
contexto, el mismo que arma extraer_fragmentos.py), detectar qué emisores
aparecen y clasificarlos en los 5 tipos del codebook:
    1. Estatal                      (a Presidente, b Área social/desarrollo,
                                     c Área económica/laboral,
                                     d Subnacional, e Otros poderes)
    2. Organizaciones y movimientos (a UTEP y sus organizaciones,
                                     b Patria Grande, c Izquierda/no alineadas,
                                     d Sindicatos tradicionales)
    3. Dirigentes sociales
    4. Sector privado / empresarial
    5. Otros / medios / academia    (incluye las notas de opinión/editoriales)

Tarea 2: para cada mención, detectar cuáles de los 16 marcos de referencia
aparecen en el mismo fragmento y cruzar emisor x marco.

Es una codificación por diccionario (búsqueda de nombres/frases, no NLP/NER
real): manual y transparente a propósito, para que se pueda auditar y
corregir el listado de patrones. Los patrones tolerantes a tildes y
mayúsculas ("Pérsico" = "persico" = "PÉRSICO"). Es un primer corte
cuantitativo, no un reemplazo de la lectura de los casos dudosos.

Uso:
    python3 analisis_emisores_marcos.py
Lee fragmentos_economia_popular.xlsx y guarda:
    - fragmentos_codificados.xlsx   (una fila por mención, con las columnas
      emisores_detectados / tipo_emisor / subtipo_emisor / marcos_detectados)
    - tabla_emisores_marcos.xlsx    (hojas: tipo_x_marco, subtipo_x_marco,
      emisor_x_marco - el resultado de la Tarea 2)
"""

import re
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------

try:
    RUTA_FRAGMENTOS = f"{CARPETA_PROYECTO}/fragmentos_economia_popular.xlsx"
except NameError:
    RUTA_FRAGMENTOS = "fragmentos_economia_popular.xlsx"

try:
    RUTA_SALIDA_FRAGMENTOS = f"{CARPETA_PROYECTO}/fragmentos_codificados.xlsx"
    RUTA_SALIDA_TABLA = f"{CARPETA_PROYECTO}/tabla_emisores_marcos.xlsx"
except NameError:
    RUTA_SALIDA_FRAGMENTOS = "fragmentos_codificados.xlsx"
    RUTA_SALIDA_TABLA = "tabla_emisores_marcos.xlsx"

SEP = " | "


# ---------------------------------------------------------------------------
# HELPERS DE PATRONES
# ---------------------------------------------------------------------------

_TILDES = {"a": "[aáà]", "e": "[eéè]", "i": "[iíì]", "o": "[oóò]", "u": "[uúü]"}


def tol(frase):
    """Frase literal -> regex tolerante a tildes y a espacios múltiples, con
    límites de palabra. 'Persico' matchea 'Pérsico'."""
    partes = []
    for ch in frase.lower():
        if ch == " ":
            partes.append(r"\s+")
        elif ch in _TILDES:
            partes.append(_TILDES[ch])
        else:
            partes.append(re.escape(ch))
    return r"\b" + "".join(partes) + r"\b"


def N(*frases):
    """Varios términos literales -> un solo patrón. Si empieza con 're:' se
    toma como regex crudo (para raíces, plurales y exclusiones)."""
    return [f[3:] if f.startswith("re:") else tol(f) for f in frases]


def compilar(patrones):
    return re.compile("|".join(patrones), re.IGNORECASE)


# ---------------------------------------------------------------------------
# TAREA 1 - EMISORES (taxonomía de Daniela)
# ---------------------------------------------------------------------------

T1 = "1. Estatal"
T2 = "2. Organizaciones y movimientos"
T3 = "3. Dirigentes sociales"
T4 = "4. Sector privado"
T5 = "5. Otros / medios / academia"

TIPOS = [T1, T2, T3, T4, T5]

S1A = "1a. Presidente"
S1B = "1b. Ejecutivo nacional - Área social/desarrollo"
S1C = "1c. Ejecutivo nacional - Área económica/laboral"
S1D = "1d. Ejecutivo subnacional"
S1E = "1e. Otros poderes del Estado"
S2A = "2a. UTEP y sus organizaciones"
S2B = "2b. Patria Grande"
S2C = "2c. Organizaciones de izquierda / no alineadas"
S2D = "2d. Sindicatos tradicionales"

# nombre canónico -> (tipo, subtipo, patrones)
EMISORES = {
    # --- 1. ESTATAL ---------------------------------------------------------
    "Presidente de la Nación": (T1, S1A, N(
        "Alberto Fernández", "Mauricio Macri", "Presidencia de la Nación",
        # "el presidente" a secas; se excluye "el presidente de la UTEP/Cámara..."
        r"re:\bel\s+presidente\b(?!\s+(?:de|del)\s+(?!la\s+naci[oó]n))",
        "Gobierno nacional", "Casa Rosada",
    )),
    "Ministerio de Desarrollo Social / Economía Social": (T1, S1B, N(
        "Daniel Arroyo", "Arroyo", "Juan Zabaleta",
        "Ministerio de Desarrollo Social", "ministro de Desarrollo Social",
        "ministra de Desarrollo Social", "Secretaría de Economía Social",
        "secretario de Economía Social", "Secretaría de Economía Popular",
        "Alexandre Roig", "INAES",
        "Instituto Nacional de Asociativismo y Economía Social",
    )),
    "Ministerio de Trabajo / Economía": (T1, S1C, N(
        "Claudio Moroni", "Moroni", "Ministerio de Trabajo",
        "ministro de Trabajo", "ministra de Trabajo",
        "Ministerio de Economía", "ministro de Economía",
        "ministra de Economía", "Ministerio de Hacienda",
    )),
    "Gobierno subnacional (provincias y municipios)": (T1, S1D, N(
        "gobernador", "gobernadora", "intendente", "intendenta",
        "jefe de Gobierno", "Axel Kicillof", "Kicillof",
        "Horacio Rodríguez Larreta", "Rodríguez Larreta",
        "Gobierno porteño", "Gobierno bonaerense", "Gobierno de la Ciudad",
        "Gobierno de la Provincia", "Municipalidad",
    )),
    "Poder Legislativo / Judicial": (T1, S1E, N(
        "Congreso", "Senado", "Cámara de Diputados", "diputado", "diputada",
        "senador", "senadora", "legislador", "legisladora", "Corte Suprema",
        "juez", "jueza", "Poder Judicial", "Poder Legislativo",
    )),

    # --- 2. ORGANIZACIONES Y MOVIMIENTOS SOCIALES / SINDICALES ---------------
    "UTEP / CTEP": (T2, S2A, N(
        "UTEP", "CTEP",
        r"re:\buni[oó]n\s+de\s+trabajadores(?:\s+y\s+trabajadoras)?\s+de\s+la\s+econom[ií]a\s+popular\b",
        r"re:\bconfederaci[oó]n\s+de\s+trabajadores(?:\s+y\s+trabajadoras)?\s+de\s+la\s+econom[ií]a\s+popular\b",
    )),
    "Movimiento Evita": (T2, S2A, N("Movimiento Evita", "Agrupación Evita")),
    "Barrios de Pie": (T2, S2A, N("Barrios de Pie", "Somos Barrios de Pie")),
    "Otras organizaciones de la UTEP": (T2, S2A, N(
        "Movimiento de Trabajadores Excluidos", "MTE",
        "Corriente Clasista y Combativa", "Frente Popular Darío Santillán",
        "Frente Darío Santillán", "Movimiento Octubres",
        "Movimiento Nacional Campesino Indígena", "MNCI",
        "Nuestra América", "Organizaciones Libres del Pueblo",
        "Federación de Organizaciones de Base", "Movimiento Popular La Dignidad",
        "Corriente Popular Eva Perón", "Misioneros de Francisco",
        "Movimiento de Inclusión Popular", "Movimiento Nacional de Empresas Recuperadas",
        "Federación Nacional Campesina", "Frente de Organizaciones Populares",
        "Frente Barrial 19 de Diciembre", "Movimiento Popular Los Pibes",
        "Federación Argentina de Cartoneros, Carreros y Recicladores", "FACCYR",
        "Movimiento Social 17 de Noviembre", "Frente Nacional Néstor Kirchner",
    )),
    "Frente Patria Grande": (T2, S2B, N("Frente Patria Grande", "Patria Grande")),
    "Organizaciones de izquierda / no alineadas": (T2, S2C, N(
        "Polo Obrero", "Movimiento Socialista de los Trabajadores", "MST",
        "Unidad Piquetera", "Barrios de Pie Libres del Sur",
        "Frente de Izquierda", "Partido Obrero",
    )),
    "Sindicatos tradicionales (CGT / CTA)": (T2, S2D, N(
        "CGT", "Confederación General del Trabajo", "CTA",
        "CTA de los Trabajadores", "CTA Autónoma", "CTA-T", "CTA-A",
    )),

    # --- 3. DIRIGENTES SOCIALES ---------------------------------------------
    # Las personas van acá (no en la organización), aunque el documento
    # también nombra a Grabois y Castro dentro de 2b.
    "Juan Grabois": (T3, "Dirigente social", N("Juan Grabois", "Grabois")),
    "Emilio Pérsico": (T3, "Dirigente social", N("Emilio Pérsico", "Pérsico")),
    "Esteban \"Gringo\" Castro": (T3, "Dirigente social", N(
        "Esteban Castro", "Gringo Castro", "Esteban \"Gringo\" Castro",
        "Esteban “Gringo” Castro",
    )),
    "Alejandro \"Peluca\" Gramajo": (T3, "Dirigente social", N(
        "Alejandro Gramajo", "Peluca Gramajo", "Gramajo",
    )),
    "Fernando \"Chino\" Navarro": (T3, "Dirigente social", N(
        "Fernando Navarro", "Chino Navarro", "Fernando Chino Navarro",
    )),
        "Daniel Menéndez": (T3, "Dirigente social", N("Daniel Menéndez", "Menéndez")),
    "Eduardo \"Chiquito\" Belliboni": (T3, "Dirigente social", N(
        "Eduardo Belliboni", "Chiquito Belliboni", "Belliboni",
    )),
    "Fernanda Miño": (T3, "Dirigente social", N("Fernanda Miño")),

    # --- 4. SECTOR PRIVADO / EMPRESARIAL ------------------------------------
    "Cámaras y entidades empresarias": (T4, "Sector privado / empresarial", N(
        "UIA", "Unión Industrial Argentina", "CAME",
        "Confederación Argentina de la Mediana Empresa", "Sociedad Rural",
        "Cámara Argentina de Comercio", "Cámara Argentina de la Construcción",
        "Coninagro", "Confederaciones Rurales", "Federación Agraria",
        "Asociación Empresaria", "cámara empresaria", "cámaras empresarias",
        "empresario", "empresaria", "empresarios", "empresarias",
        "Consejo Interamericano de Comercio","cámara empresarial", "cámaras empresariales",
    )),

    # --- 5. OTROS / MEDIOS / ACADEMIA ---------------------------------------
    "Expertos y academia": (T5, "Academia / expertos", N(
        "especialista", "especialistas", "investigador", "investigadora",
        "investigadores", "economista", "economistas", "sociólogo",
        "socióloga", "politólogo", "politóloga", "académico", "académica",
        "analista", "analistas", "CONICET", "Universidad", "consultora",
        "Observatorio",
    )),
    "Iglesia / Papa Francisco": (T5, "Otros", N(
        "Papa Francisco", "Iglesia", "Pastoral Social", "Cáritas",
    )),
}

# Emisor "de prensa": se detecta por la sección o la URL de la nota, no por
# el texto del fragmento.
EMISOR_EDITORIAL = "Nota de opinión / editorial"
EMISORES[EMISOR_EDITORIAL] = (T5, "Medios / notas editoriales", [])
PATRON_EDITORIAL = re.compile(r"opini[oó]n|editorial|columnista|/opinion/", re.IGNORECASE)

EMISORES_COMPILADOS = {
    nombre: compilar(pats) for nombre, (_t, _s, pats) in EMISORES.items() if pats
}


# ---------------------------------------------------------------------------
# TAREA 2 - MARCOS DE REFERENCIA (los 16 de Daniela)
# ---------------------------------------------------------------------------
# Los marcos 12 a 16 son categorías "paraguas": se detectan por cualquiera de
# los términos que el documento lista en "Comprende".

MARCOS = {
    "01. Organizaciones de la economía popular": N(
        "organizaciones de la economía popular", "organización de la economía popular",
        "organizaciones sociales", "organización social", "organizaciones populares",
        "organizaciones de la economía social", "organización de la economía social",
    ),
    "02. Movimientos sociales": N(
        "movimientos sociales", "movimiento social", "movimientos populares",
    ),
    "03. Sector": [
        # "sector" suelto, sin los usos que no son el sector de la EP
        r"\bsector(?:es)?\b(?!\s+(?:privado|p[uú]blico|agropecuario|financiero|energ[eé]tico|empresari\w+|industrial|exportador))",
    ],
    "04. Políticas sociales y laborales": N(
        "política social", "políticas sociales", "política laboral",
        "políticas laborales", "política de empleo", "políticas de empleo",
        "políticas públicas", "política pública",
    ),
    "05. Trabajadores informales": N(
        "trabajador informal", "trabajadores informales", "trabajadora informal",
        "trabajadoras informales", "informales",
        "desocupado", "desocupados", "desocupada", "desocupadas", "sin empleo",
"trabajador en negro", "trabajadores en negro", "trabajadora en negro", "trabajadoras en negro",
    ),
    "06. Pobres": [r"\bpobres?\b", r"\bpobreza\b", r"\bindigen\w+"],
    "07. Humildes": [r"\bhumildes?\b", r"\blos\s+m[aá]s\s+necesitados\b", r"\bsectores?\s+m[aá]s\s+postergados?\b"],
    "08. Piqueteros": [r"\bpiquetero\w*"],
    "09. Derechos": [r"\bderechos\b"],
    "10. IFE": N("IFE", "Ingreso Familiar de Emergencia"),
    "11. Potenciar Trabajo": N("Potenciar Trabajo"),
    "12. Trabajo / trabajadores": N(
        "trabajador por cuenta propia", "trabajadores por cuenta propia",
        "trabajadora por cuenta propia", "trabajador de la economía popular",
        "trabajadores de la economía popular", "trabajadora de la economía popular",
        "trabajadores y trabajadoras de la economía popular",
        "trabajador de la economía social", "trabajadores de la economía social",
        "productor de la economía popular", "productores de la economía popular",
        "agricultor familiar", "agricultores familiares", "agricultura familiar",
        "cartonero", "cartoneros", "cartonera", "cartoneras",
        "cooperativa", "cooperativas", "unidad productiva", "unidades productivas",
        "autogestión", "economía social y solidaria", "economía social",
        "economía solidaria", "inventan su propio trabajo", "inventores de su propio trabajo",
        "changarín", "changarines", "trabajador golondrina", "trabajadores golondrina",
        "artesano", "artesanos", "artesana", "artesanas",
        "empresa recuperada", "empresas recuperadas", "fábrica recuperada",
        "fábricas recuperadas",
        # "trabajador(es)" suelto, pero sin contar el nombre de la UTEP/CTEP
        # ("Unión/Confederación de Trabajadores de la Economía Popular")
        r"re:(?<!\bde\s)\btrabajador(?:es|a|as)?\b(?!\s+(?:y\s+trabajadoras\s+)?de\s+la\s+econom)",
        "vendedores ambulantes", "vendedor ambulante", "vendedora ambulante",
        r"re:\btrabajador(?:es|a|as)?\s+(?:de|en)\s+(?:los\s+)?(?:comedor|merendero)\w*",

    ),
    "13. Asistencia social": N(
        "plan social", "planes sociales", "Potenciar Trabajo", "subsidio",
        "subsidios", "asistencia", "asistencial", "asistencialismo", "contención",
        "tarjeta alimentar", "Tarjeta Alimentar", "comedor", "comedores",
        "merendero", "merenderos", "vulnerabilidad", "población vulnerable",
        "ayuda del Estado", "ayudas del Estado","beneficiario", "beneficiarios", "beneficiaria", "beneficiarias",
        r"re:\btitulares?\s+de\s+(?:programas|planes)\s+sociales\b",
    ),
    "14. Informalidad / precarización": N(
        "trabajo no registrado", "trabajo en negro", "trabajador informal",
        "trabajadores informales", "informalidad", "informalidad laboral",
        "marginalidad", "economía no formal", "economía informal", "changa",
        "changas", "precariedad laboral", "precarización", "precariedad",
        "subempleo", "venta ambulante", "vendedores ambulantes",
        "vendedor ambulante", "vendedora ambulante",
        "desocupación",
    ),
    "15. Conflicto / seguridad / orden público": [
        r"\bpiquet\w*", r"\bprotestas?\b", r"\bmovilizaci[oó]n(?:es)?\b",
        r"\bacampes?\b", r"\bintermediaci[oó]n\b", r"(?<!en\s)\bmarchas?\b",
        r"\bocupaci[oó]n(?:es)?\b", r"\bcortes?\s+de\s+(?:calle|ruta)s?\b",
        r"\bmanifestaci[oó]n(?:es)?\b",
    ],
    "16. Integración económica / desarrollo productivo": N(
        "RENATEP", "Registro Nacional de Trabajadores de la Economía Popular",
        "monotributo social", "salario social complementario",
        "acceso al crédito", "crédito", "créditos", "compras estatales",
        "compre estatal", "cadena de valor", "cadenas de valor",
        "inclusión financiera", "herramientas productivas",
        "desarrollo productivo", "fondo de capital social",
    ),
}

MARCOS_COMPILADOS = {nombre: compilar(pats) for nombre, pats in MARCOS.items()}


# ---------------------------------------------------------------------------
# DETECCIÓN
# ---------------------------------------------------------------------------

def _es_editorial(fila):
    texto = " ".join(str(fila.get(c) or "") for c in ("seccion", "link"))
    return bool(PATRON_EDITORIAL.search(texto))


# Nombres propios que ya contienen el término "economía popular": si solo aparecen como
# nombre completo, no cuentan como emisor (la sigla UTEP/CTEP u otro nombre sí).
PATRON_NOMBRE_CON_TERMINO = re.compile(
    r"(?:uni[oó]n|confederaci[oó]n)\s+de\s+trabajadores(?:\s+y\s+trabajadoras)?\s+de\s+la\s+econom[ií]as?\s+popular(?:es)?"
    r"|registro\s+nacional\s+de\s+trabajadores(?:\s+y\s+trabajadoras)?\s+de\s+la\s+econom[ií]as?\s+popular(?:es)?"
    r"|secretar[ií]a\s+de\s+econom[ií]a\s+popular",
    re.IGNORECASE,
)


def detectar_emisores(fragmento, fila=None):
    """Devuelve la lista de emisores encontrados en el fragmento (más el
    emisor editorial si la nota es de opinión). El nombre completo de una
    organización que contiene el término no cuenta como emisor por sí solo."""
    encontrados = []
    if isinstance(fragmento, str):
        limpio = PATRON_NOMBRE_CON_TERMINO.sub(" ", fragmento)
        encontrados = [n for n, p in EMISORES_COMPILADOS.items() if p.search(limpio)]
    if fila is not None and _es_editorial(fila):
        encontrados.append(EMISOR_EDITORIAL)
    return encontrados


def tipos_y_subtipos(emisores):
    tipos = [t for t in TIPOS if any(EMISORES[e][0] == t for e in emisores)]
    subtipos = sorted({EMISORES[e][1] for e in emisores})
    return tipos, subtipos


# El marco 12 (Trabajo / trabajadores) se busca sin los nombres propios que contienen
# "Trabajadores de la Economía Popular" (UTEP, CTEP, RENATEP): si no, el nombre de la
# organización activa el marco aunque la nota no hable de trabajo. Los demás marcos se
# buscan sobre el fragmento completo (el 16 usa el nombre del RENATEP a propósito).
MARCOS_SIN_NOMBRES = {"12. Trabajo / trabajadores"}


def detectar_marcos(fragmento):
    if not isinstance(fragmento, str):
        return []
    limpio = PATRON_NOMBRE_CON_TERMINO.sub(" ", fragmento)
    return [n for n, p in MARCOS_COMPILADOS.items()
            if p.search(limpio if n in MARCOS_SIN_NOMBRES else fragmento)]


def codificar(df):
    filas = []
    for _, fila in df.iterrows():
        emisores = detectar_emisores(fila.get("fragmento"), fila)
        tipos, subtipos = tipos_y_subtipos(emisores)
        marcos = detectar_marcos(fila.get("fragmento"))
        filas.append({
            **fila.to_dict(),
            "emisores_detectados": SEP.join(emisores) if emisores else None,
            "tipo_emisor": SEP.join(tipos) if tipos else None,
            "subtipo_emisor": SEP.join(subtipos) if subtipos else None,
            "marcos_detectados": SEP.join(marcos) if marcos else None,
        })
    return pd.DataFrame(filas)


def _tiene(serie, valor):
    """Serie booleana: la celda (lista unida con ' | ') contiene `valor`
    como elemento exacto."""
    return serie.fillna("").apply(lambda s: valor in s.split(SEP)).astype(bool)


def _cruce(df_cod, columna, valores, extra=None):
    filas = []
    for v in valores:
        sub = df_cod[_tiene(df_cod[columna], v)]
        fila = {columna: v, **(extra(v) if extra else {}), "total_menciones": len(sub)}
        for marco in MARCOS:
            fila[marco] = int(_tiene(sub["marcos_detectados"], marco).sum())
        filas.append(fila)
    return pd.DataFrame(filas)


def tabla_cruce(df_cod):
    """Tarea 2. Devuelve tres tablas emisor x marco, a distinto nivel:
    por tipo, por subtipo y por emisor individual."""
    subtipos = [s for s in dict.fromkeys(EMISORES[e][1] for e in EMISORES)]
    por_tipo = _cruce(df_cod, "tipo_emisor", TIPOS)
    por_subtipo = _cruce(
        df_cod, "subtipo_emisor", subtipos,
        extra=lambda s: {"tipo": next(EMISORES[e][0] for e in EMISORES if EMISORES[e][1] == s)},
    )
    por_emisor = _cruce(
        df_cod, "emisores_detectados", list(EMISORES),
        extra=lambda e: {"tipo": EMISORES[e][0], "subtipo": EMISORES[e][1]},
    )
    por_emisor = por_emisor.rename(columns={"emisores_detectados": "emisor"})
    por_subtipo = por_subtipo.rename(columns={"subtipo_emisor": "subtipo"})
    por_tipo = por_tipo.rename(columns={"tipo_emisor": "tipo"})
    return por_tipo, por_subtipo, por_emisor


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

print(f"Leyendo fragmentos desde: {RUTA_FRAGMENTOS}")
df = pd.read_excel(RUTA_FRAGMENTOS)

df_codificado = codificar(df)
n = len(df_codificado)
con_emisor = df_codificado["emisores_detectados"].notna().sum()
con_marco = df_codificado["marcos_detectados"].notna().sum()
print(f"\nTotal de menciones: {n}")
print(f"Con al menos un emisor detectado: {con_emisor} ({con_emisor / n:.1%})")
print(f"Con al menos un marco detectado: {con_marco} ({con_marco / n:.1%})")

df_codificado.to_excel(RUTA_SALIDA_FRAGMENTOS, index=False)
print(f"Guardado: {RUTA_SALIDA_FRAGMENTOS}")

por_tipo, por_subtipo, por_emisor = tabla_cruce(df_codificado)

print("\nMenciones por tipo de emisor:")
print(por_tipo[["tipo", "total_menciones"]].to_string(index=False))
print("\nMenciones por emisor:")
print(por_emisor.sort_values("total_menciones", ascending=False)[
    ["emisor", "tipo", "total_menciones"]].to_string(index=False))
print("\nMenciones por marco:")
print(pd.Series({m: int(_tiene(df_codificado["marcos_detectados"], m).sum())
                 for m in MARCOS}).sort_values(ascending=False).to_string())

print("\n" + "=" * 60)
print("TAREA 2 - cruce tipo de emisor x marco")
print("=" * 60)
with pd.ExcelWriter(RUTA_SALIDA_TABLA) as w:
    por_tipo.to_excel(w, sheet_name="tipo_x_marco", index=False)
    por_subtipo.to_excel(w, sheet_name="subtipo_x_marco", index=False)
    por_emisor.to_excel(w, sheet_name="emisor_x_marco", index=False)
print(f"Guardado: {RUTA_SALIDA_TABLA}")
print(por_tipo.to_string(index=False))

print(
    "\nOjo: esto es codificación por diccionario, no NLP real. Los patrones "
    "genéricos (sector, derechos, asistencia, marcha, trabajador, "
    "empresarios, Universidad, Francisco, 'el presidente') pueden traer "
    "falsos positivos - conviene revisar a mano los casos de mayor volumen "
    "antes de citarlos como definitivos."
)

## Paso 7 - Gráficos de los resultados

Cuatro figuras que **solo se muestran acá** (no se guardan archivos):

- **Menciones por emisor** (top 15), coloreado por tipo de emisor.
- **Menciones por marco de referencia** (los 16).
- **Mapa de calor tipo de emisor x marco**: el número es el conteo y el color el % de las menciones de ese tipo.
- **Mapa de calor de los emisores más mencionados x marco**.

In [ ]:
"""
Gráficos de los resultados de Tarea 1 y Tarea 2 (emisores y marcos de
referencia de "economía popular"), con la taxonomía de Daniela.

Lee fragmentos_codificados.xlsx y tabla_emisores_marcos.xlsx (las salidas
de analisis_emisores_marcos.py) y MUESTRA (no guarda) cuatro figuras:

1. Menciones por emisor (top 15), coloreado por tipo de emisor.
2. Menciones por marco de referencia (los 16).
3. Mapa de calor tipo de emisor x marco (Tarea 2) - en % de las menciones de
   cada tipo, con el conteo en cada celda.
4. Mapa de calor de los emisores más mencionados x marco (Tarea 2).

Uso:
    python3 graficos_resultados.py
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------

try:
    RUTA_FRAGMENTOS = f"{CARPETA_PROYECTO}/fragmentos_codificados.xlsx"
    RUTA_TABLA = f"{CARPETA_PROYECTO}/tabla_emisores_marcos.xlsx"
except NameError:
    RUTA_FRAGMENTOS = "fragmentos_codificados.xlsx"
    RUTA_TABLA = "tabla_emisores_marcos.xlsx"

SEP = " | "

# Un color por tipo de emisor (categórica, en orden fijo, no se cicla).
COLOR_POR_TIPO = {
    "1. Estatal": "#2a78d6",                          # azul
    "2. Organizaciones y movimientos": "#eb6834",     # naranja
    "3. Dirigentes sociales": "#1baf7a",              # aqua
    "4. Sector privado": "#eda100",                   # amarillo
    "5. Otros / medios / academia": "#e87ba4",        # magenta
}
RAMPA_SECUENCIAL = [
    "#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
    "#256abf", "#184f95", "#104281", "#0d366b",
]
TINTA_PRIMARIA = "#0b0b0b"
TINTA_SECUNDARIA = "#52514e"
TINTA_MUTED = "#898781"
GRILLA = "#e1e0d9"
SUPERFICIE = "#fcfcfb"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "figure.facecolor": SUPERFICIE,
    "axes.facecolor": SUPERFICIE,
    "axes.edgecolor": GRILLA,
    "text.color": TINTA_PRIMARIA,
    "axes.labelcolor": TINTA_SECUNDARIA,
    "xtick.color": TINTA_MUTED,
    "ytick.color": TINTA_MUTED,
})


def _tiene(serie, valor):
    return serie.fillna("").apply(lambda s: valor in s.split(SEP)).astype(bool)


def _columnas_marco(df):
    return [c for c in df.columns if c[:2].isdigit()]


def _estilo_barras(ax, titulo):
    ax.set_title(titulo, fontsize=13, color=TINTA_PRIMARIA, loc="left", pad=12)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRILLA)
    ax.xaxis.grid(True, color=GRILLA, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)


def _etiquetas_valor(ax, barras, valores):
    for barra, valor in zip(barras, valores):
        ax.text(valor + max(valores) * 0.01, barra.get_y() + barra.get_height() / 2,
                str(int(valor)), va="center", ha="left", fontsize=9, color=TINTA_SECUNDARIA)


# ---------------------------------------------------------------------------
# GRÁFICOS SEPARADOS POR DIARIO (Página/12 y La Nación) - reemplaza a los anteriores
# Las tablas por diario se recalculan desde df_frag (fragmentos_codificados.xlsx),
# que ya trae la columna "medio"; tabla_emisores_marcos.xlsx solo se usa para el
# listado de tipos/emisores y los nombres de los marcos.
# ---------------------------------------------------------------------------

DIARIOS = ["Página/12", "La Nación"]
COLOR_DIARIO = {"Página/12": "#d6342f", "La Nación": "#2a78d6"}   # Página/12 rojo, La Nación azul

def _tabla_diario(df_frag, columna, valores, marcos, medio):
    """Filas = valores (tipos o emisores), columnas = total_menciones + un conteo por marco,
    solo con las menciones de ese diario."""
    sub = df_frag[df_frag["medio"] == medio]
    filas = {}
    for v in valores:
        s = sub[_tiene(sub[columna], v)]
        fila = {"total_menciones": len(s)}
        for m in marcos:
            fila[m] = int(_tiene(s["marcos_detectados"], m).sum())
        filas[v] = fila
    return pd.DataFrame.from_dict(filas, orient="index")


# ---------------------------------------------------------------------------
# GRÁFICO 1 - menciones por emisor (top N), un panel por diario
# ---------------------------------------------------------------------------

def grafico_emisores(df_frag, df_emisor, top=15, por_nota=True):
    """por_nota=True cuenta en cuántas NOTAS aparece cada emisor (una vez por nota); así una entrevista
    larga con 13 menciones no suma 13 veces a Grabois. por_nota=False cuenta menciones (fragmentos)."""
    tipo_de = df_emisor.set_index("emisor")["tipo"]
    unidad = "notas" if por_nota else "menciones (fragmentos)"

    def contar(sub, e):
        s = sub[_tiene(sub["emisores_detectados"], e)]
        return int(s["link"].nunique()) if por_nota else len(s)

    conteos, n_medio = {}, {}
    for d in DIARIOS:
        sub = df_frag[df_frag["medio"] == d]
        conteos[d] = pd.Series({e: contar(sub, e) for e in tipo_de.index})
        n_medio[d] = sub["link"].nunique() if por_nota else len(sub)
    xmax = max(conteos[d].max() for d in DIARIOS) * 1.15

    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    for ax, d in zip(axs, DIARIOS):
        s = conteos[d][conteos[d] > 0].sort_values(ascending=False).head(top).sort_values()
        barras = ax.barh(s.index, s.values, color=[COLOR_POR_TIPO[tipo_de[e]] for e in s.index], height=0.65)
        for b, v in zip(barras, s.values):
            ax.text(v + xmax * 0.01, b.get_y() + b.get_height() / 2, str(int(v)),
                    va="center", fontsize=9, color=TINTA_SECUNDARIA)
        ax.set_xlim(0, xmax)
        ax.set_xlabel(f"{unidad.capitalize()} en que aparece el emisor")
        _estilo_barras(ax, f"{d} (n={n_medio[d]} {'notas' if por_nota else 'menciones'})")
        for t in (ax.title, ax._left_title):       # el título lleva el color del diario
            t.set_color(COLOR_DIARIO[d])
    tipos_pres = [t for t in COLOR_POR_TIPO if t in set(tipo_de)]
    fig.legend(handles=[Patch(facecolor=COLOR_POR_TIPO[t], label=t) for t in tipos_pres],
               loc="lower center", ncol=len(tipos_pres), frameon=False, fontsize=9)
    fig.suptitle(f"Emisores más frecuentes por diario (top {top}; unidad: {unidad})", x=0.01, ha="left", fontsize=13)
    fig.tight_layout(rect=(0, 0.06, 1, 0.97))
    plt.show()


# ---------------------------------------------------------------------------
# GRÁFICO 2 - menciones por marco, una barra por diario (en % de las menciones del diario)
# ---------------------------------------------------------------------------

def grafico_marcos(df_frag, marcos):
    n_medio = df_frag["medio"].value_counts()
    cont = pd.DataFrame({d: {m: int(_tiene(df_frag[df_frag["medio"] == d]["marcos_detectados"], m).sum())
                             for m in marcos} for d in DIARIOS})
    cont = cont.loc[cont.sum(axis=1).sort_values().index]      # orden por total, de menor a mayor
    y = np.arange(len(cont)); h = 0.38

    fig, ax = plt.subplots(figsize=(10, 7.5))
    for i, d in enumerate(DIARIOS):
        pct = cont[d] / max(n_medio.get(d, 1), 1) * 100
        barras = ax.barh(y + (i - 0.5) * h, pct, h, color=COLOR_DIARIO[d], label=f"{d} (n={n_medio.get(d, 0)})")
        for b, v, c in zip(barras, pct, cont[d]):
            ax.text(v + 0.4, b.get_y() + b.get_height() / 2, str(int(c)), va="center", fontsize=8, color=TINTA_SECUNDARIA)
    ax.set_yticks(y); ax.set_yticklabels(cont.index, fontsize=9)
    ax.set_xlabel("% de las menciones del diario (el número es el conteo)")
    _estilo_barras(ax, "Menciones por marco de referencia, por diario")
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
# GRÁFICOS 3 y 4 - mapas de calor emisor x marco, un panel por diario
# ---------------------------------------------------------------------------
# Rampa de cada diario: Página/12 en gradiente rojo, La Nación en gradiente azul (el de siempre)
RAMPA_ROJA = ["#fcfcfb", "#fbdcd8", "#f5b5ae", "#ee8c83", "#e4635a", "#c9423a", "#a02f2a", "#76221f"]
RAMPA_DIARIO = {"Página/12": RAMPA_ROJA, "La Nación": RAMPA_SECUENCIAL}

def _heatmap_doble(tablas, marcos, titulo, figsize=(17, 5.5)):
    """tablas: {diario: DataFrame filas x (total_menciones + marcos)}. El color es el % de las
    menciones de la fila EN ESE DIARIO; el número en la celda es el conteo."""
    filas = list(tablas[DIARIOS[0]].index)
    pct = {}
    for d in DIARIOS:
        t = tablas[d].loc[filas]
        pct[d] = t[marcos].values.astype(float) / np.maximum(t["total_menciones"].values[:, None].astype(float), 1) * 100
    vmax = max(p.max() for p in pct.values()) or 1

    fig, axs = plt.subplots(1, 2, figsize=figsize, sharey=True, layout="constrained")
    for ax, d in zip(axs, DIARIOS):
        t = tablas[d].loc[filas]
        cmap = LinearSegmentedColormap.from_list("rampa_" + d, RAMPA_DIARIO[d])
        im = ax.imshow(pct[d], cmap=cmap, vmin=0, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(marcos)))
        ax.set_xticklabels(marcos, rotation=45, ha="right", fontsize=8.5)
        ax.set_yticks(range(len(filas)))
        ax.set_yticklabels([f"{f}  (n={int(t.loc[f, 'total_menciones'])})" for f in filas], fontsize=9)
        for i in range(len(filas)):
            for j, m in enumerate(marcos):
                v = int(t.iloc[i][m])
                if v:
                    ax.text(j, i, str(v), ha="center", va="center", fontsize=8,
                            color="#ffffff" if pct[d][i, j] > vmax * 0.55 else TINTA_PRIMARIA)
        ax.set_title(d, fontsize=12, loc="left", color=COLOR_DIARIO[d], pad=10)
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.set_xticks(np.arange(-0.5, len(marcos), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(filas), 1), minor=True)
        ax.grid(which="minor", color=SUPERFICIE, linewidth=2)
        ax.tick_params(which="minor", length=0)
        cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)      # una barra de color por diario
        cbar.set_label("% de las menciones de la fila", color=TINTA_SECUNDARIA, fontsize=9)
        cbar.ax.tick_params(labelsize=8, color=TINTA_MUTED, labelcolor=TINTA_MUTED)
        cbar.outline.set_visible(False)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=13, color=TINTA_PRIMARIA)
    plt.show()

def grafico_heatmap_tipo(df_frag, df_tipo):
    marcos = _columnas_marco(df_tipo)
    tipos = list(df_tipo["tipo"])
    tablas = {d: _tabla_diario(df_frag, "tipo_emisor", tipos, marcos, d) for d in DIARIOS}
    _heatmap_doble(tablas, marcos, "Tipo de emisor x marco de referencia (color = % de las menciones del tipo en cada diario)",
                   figsize=(17, 4.8))


def grafico_heatmap_emisores(df_frag, df_emisor, top=12):
    marcos = _columnas_marco(df_emisor)
    emisores = list(df_emisor.sort_values("total_menciones", ascending=False).head(top)["emisor"])  # mismos emisores en los dos paneles
    tablas = {d: _tabla_diario(df_frag, "emisores_detectados", emisores, marcos, d) for d in DIARIOS}
    _heatmap_doble(tablas, marcos, "Emisores más mencionados x marco de referencia (color = % de las menciones del emisor en cada diario)",
                   figsize=(17, 6.5))


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

print(f"Leyendo: {RUTA_FRAGMENTOS}")
df_frag = pd.read_excel(RUTA_FRAGMENTOS)
print(f"Leyendo: {RUTA_TABLA}")
df_tipo = pd.read_excel(RUTA_TABLA, sheet_name="tipo_x_marco")
df_emisor = pd.read_excel(RUTA_TABLA, sheet_name="emisor_x_marco")
print(df_frag["medio"].value_counts().to_string())

grafico_emisores(df_frag, df_emisor)
grafico_marcos(df_frag, _columnas_marco(df_tipo))
grafico_heatmap_tipo(df_frag, df_tipo)
grafico_heatmap_emisores(df_frag, df_emisor)

In [ ]:
print(df_frag.groupby("medio")["link"].nunique())